# Style Transfer Training on Kaggle

**Before running any cell**, do these two things in the Kaggle notebook UI (no code required):

1. **Add COCO 2017 dataset**  
   Notebook sidebar → *+ Add Data* → search **`awsaf49/coco-2017-dataset`** → Add.  
   This mounts the dataset at `/kaggle/input/datasets/awsaf49/coco-2017-dataset/` — no download needed.

2. **Enable GPU**  
   Notebook sidebar → *Settings* → Accelerator → **GPU T4 x1**.

Then run the cells below top-to-bottom.


## Step 1 — Verify GPU and COCO images are available

In [ ]:
import os
import pathlib

# ── GPU check ────────────────────────────────────────────────────────────────
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# ── COCO images check ────────────────────────────────────────────────────────
COCO_TRAIN = pathlib.Path("/kaggle/input/datasets/awsaf49/coco-2017-dataset/coco2017/train2017")
imgs = list(COCO_TRAIN.glob("*.jpg"))
print(f"COCO train images found: {len(imgs):,}")   # expect ~118,287
assert len(imgs) > 1000, "Dataset not mounted — add awsaf49/coco-2017-dataset in notebook settings"


## Step 1b — Verify internet access

**Internet must be enabled** for training to work — VGG16 weights (~550 MB) are downloaded on the first run.

Enable it in: *Notebook sidebar → Settings → Internet → On*  
Then re-run this cell. If the check passes, continue to Step 2.

In [ ]:
import urllib.request

try:
    urllib.request.urlopen("https://api.github.com", timeout=5)
    print("✓ Internet is available — safe to continue.")
except Exception:
    raise RuntimeError(
        "\n\n"
        "  ✗ No internet access detected.\n\n"
        "  Training requires internet to download VGG16 weights (~550 MB).\n"
        "  Fix: Notebook sidebar → Settings → Internet → On\n"
        "  Then re-run this cell before proceeding.\n"
    )

## Step 2 — Clone the repo and install trainer dependencies

In [ ]:
import subprocess, sys, os

REPO_URL = "https://github.com/PeterWazinski/style_transfer.git"
REPO_DIR = pathlib.Path("/kaggle/working/style_transfer")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull"], check=True)

os.chdir(REPO_DIR)

# Install trainer deps (torch is already present on Kaggle GPU images)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[trainer]"], check=True)
print("Install done. Working dir:", os.getcwd())


## Step 3 — Upload your style reference image

Upload the style image you want to train via the Kaggle notebook UI:
*File browser (left panel) → Upload → select your JPEG*.

It will land at `/kaggle/working/<filename>`. Set `STYLE_IMAGE` below accordingly.

In [ ]:
import sys as _sys
_sys.path.insert(0, str(REPO_DIR))
from training.kaggle_training_helper import TrainingConfig, KaggleStyleRunner

# ── Edit these three lines to match your job ─────────────────────────────────
STYLE_IMAGE = pathlib.Path("/kaggle/working/style_transfer/sample_images/style-pics/hundertwasser2.jpg")   # ← your uploaded file
STYLE_ID    = "hundertwasser"     # slug used as folder name and catalog ID
STYLE_NAME  = "Hundertwasser"     # display name shown in the gallery
# ─────────────────────────────────────────────────────────────────────────────

assert STYLE_IMAGE.exists(), f"Style image not found: {STYLE_IMAGE}\nUpload it via the file browser first."

cfg = TrainingConfig(
    style_images=[STYLE_IMAGE],
    style_id=STYLE_ID,
    style_name=STYLE_NAME,
    coco_path=COCO_TRAIN,
    style_weight=1e10,   # fixed: matches yakhyo reference (do not lower)
    content_weight=1e5,  # override if desired, e.g. 5e4 for more style influence
)
runner = KaggleStyleRunner(cfg)
runner.analyse_style()


## Step 3b — Smoke test (~10 min on T4) ✅ run this before the 3-hour training

Trains for **2000 batches** only, exports a quick model, and measures how strongly
the style is applied to a **neutral content photo** (a COCO val image — NOT the style image itself).

> ⚠ **Why not test on the style image?** Testing the model on its own training target always gives a
> high pixel-change score even when the model is weak — this is a false positive. A neutral content
> photo gives the true indicator of whether the style will be visible on real photos.

| Mean pixel change on content photo | Verdict | Action |
|---|---|---|
| < 8 | ⚠ Weak | Multiply `STYLE_WEIGHT` × 10 in Step 3, re-run smoke test |
| 8 – 20 | ～ Moderate | Try `STYLE_WEIGHT` × 3 for stronger effect |
| > 20 | ✓ Good | Proceed to Step 4 |

**Local smoke test (no Kaggle needed):** You can also run a quick local test using `sample_images/`
as a tiny dataset (only ~10–20 images, trains in ~1 min on CPU):
```bash
python main_style_trainer.py train \
    --style  sample_images/style-pics/my_style.jpg \
    --coco   sample_images/ \
    --out    /tmp/smoke_test \
    --max-batches 50 \
    --style-weight 1e10 \
    --device cpu
```
Then inspect `preview.jpg` in `/tmp/smoke_test/`. The result will be over-stylised (tiny dataset
overfits fast) but gives a quick visual of whether the style direction looks correct.
If the output barely differs from the input, increase `--style-weight` by 10× before Kaggle training.


In [ ]:
cfg.smoke_batches = 2000  # 2000 validated on T4 (mean_diff=57 on candy); lower to 500 for a quicker check
# cfg.content_weight = 5e4  # reduce for stronger style influence

results = runner.run_smoke_test()

# ── 3-way display: content | styled output | style reference ─────────────────
import io as _io
import ipywidgets as _widgets
from IPython.display import display as _display

def _to_widget(pil_img, width=210):
    buf = _io.BytesIO()
    pil_img.save(buf, format="PNG")
    return _widgets.Image(value=buf.getvalue(), format="png",
                          layout=_widgets.Layout(width=f"{width}px"))

_w_content = _widgets.VBox([
    _widgets.Label("Content photo"),
    _to_widget(results["content_img"]),
])
_w_styled = _widgets.VBox([
    _widgets.Label("Styled output"),
    _to_widget(results["styled_img"]),
])
_w_ref = _widgets.VBox([
    _widgets.Label("Style reference"),
    _to_widget(results["style_ref_img"]),
])
_display(_widgets.HBox([_w_content, _w_styled, _w_ref]))


## Step 4 — Run training

Typical wall-clock times on a Kaggle T4:
| Epochs | COCO images | ~Time |
|---|---|---|
| 1 | 118 k | ~1.5 h |
| 2 | 118 k | ~3 h |

Kaggle gives you **30 h/week** GPU, so 2 epochs comfortably fits in one session.

In [ ]:
# Launches full training (~3 h on T4 × 2 for 2 epochs = 166 k images).
# Saves config.json next to model.pth — resume_training() loads it automatically.
runner.run_full_training()

## Step 5 — Check preview and download the model

After training finishes you should have `model.onnx`, `model.pth`, and `preview.jpg` in the output directory.

Download everything via: Kaggle file browser → right-click on the output folder → *Download*.

Or zip and use the cell below to make a single download link.

In [ ]:
# Copies output files to /kaggle/output/<style_id>/ and creates a zip.
# Download from the Output tab (right panel) in the Kaggle notebook UI.
runner.package_output()

## Step 6 — Resume training from last checkpoint (if interrupted)

If training was interrupted, run this cell to continue from the last saved checkpoint.  
Checkpoint files are saved every 5 000 images as `model.ckpt_NNNNN.pth` inside the output folder.

The cell automatically picks the **most recent** checkpoint so you do not need to edit anything — just run it.


In [ ]:
# Resume training from the last checkpoint.
# Reads config.json automatically — no need to re-enter weights.
# STYLE_ID and runner must be configured (run Step 3 first if they are not).
runner.resume_training()